In [1]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
import numpy as np
import pandas as pd


TARGET_COL   = "diagnosis"
TARGET_POS   = "M"
DROP_COLS    = ["id", "Unnamed: 32"]
N_SEEDS      = 30
N_SPLITS_CV  = 10
N_REPEATS_CV = 10
CV_SEED      = 42

RADIUS_BASED_DROP = [
    "area_mean",    "area_se",    "area_worst",
    "perimeter_mean", "perimeter_se", "perimeter_worst",
    "concavity_mean", "concavity_se", "concavity_worst",
]

df = pd.read_csv("Data for Task 1.csv")
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
y_str = df[TARGET_COL]
y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL]).drop(columns=RADIUS_BASED_DROP)
print(f"Radius-based feature set: {len(X.columns)} features")
print(f"  {list(X.columns)}\n")

thresholds = np.arange(0.25, 0.51, 0.01)

cv = RepeatedStratifiedKFold(
    n_splits=10,
    n_repeats=10,
    random_state=42
)

results = []

for th in thresholds:

    fold_f1 = []
    fold_recall = []
    fold_fn = []
    fold_fp = []

    for train_idx, test_idx in cv.split(X, y):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc = scaler.transform(X_test)

        lr = LogisticRegression(
            C=1,
            solver="lbfgs",
            max_iter=10000,
            class_weight="balanced",
            random_state=42
        )

        lr.fit(X_train_sc, y_train)

        y_prob = lr.predict_proba(X_test_sc)[:, 1]
        y_pred = np.where(y_prob >= th, "M", "B")

        cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

        fold_f1.append(
            f1_score(y_test, y_pred, pos_label="M")
        )

        fold_recall.append(
            recall_score(y_test, y_pred, pos_label="M")
        )

        fold_fn.append(cm[1, 0])
        fold_fp.append(cm[0, 1])

    results.append({
        "threshold": round(th, 2),
        "mean_f1": np.mean(fold_f1),
        "mean_recall": np.mean(fold_recall),
        "mean_fn": np.mean(fold_fn),
        "max_fn": np.max(fold_fn),
        "fn_ge_2": np.sum(np.array(fold_fn) >= 2),
        "mean_fp": np.mean(fold_fp)
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ["mean_fn", "mean_f1"],
    ascending=[True, False]
)

print(results_df.to_string(index=False))

Radius-based feature set: 21 features
  ['radius_mean', 'texture_mean', 'smoothness_mean', 'compactness_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'smoothness_se', 'compactness_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'smoothness_worst', 'compactness_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']

 threshold  mean_f1  mean_recall  mean_fn  max_fn  fn_ge_2  mean_fp
      0.25 0.937900     0.976861     0.49       2        6     2.31
      0.26 0.938814     0.976385     0.50       2        6     2.25
      0.28 0.942701     0.975931     0.51       2        7     2.05
      0.27 0.941238     0.975931     0.51       2        7     2.12
      0.30 0.944502     0.974978     0.53       2        7     1.94
      0.29 0.943036     0.974978     0.53       2        7     2.01
      0.31 0.945462     0.974502     0.54       2        8     1.88
      0.34 0.